In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [2]:
# notebooks/test_pipeline.ipynb
# ═══════════════════════════════════════════════════════════
# Test Pipeline with Chunked Collection
# ═══════════════════════════════════════════════════════════



from src.vector_store_chunked import get_chunked_collection
from scripts.rag_pipeline import run_pipeline, print_results
from src.memory import ConversationMemory
from src.data_loader import load_episodes

# ═══════════════════════════════════════════════════════════
# Load Resources
# ═══════════════════════════════════════════════════════════

print("Loading resources...")

collection = get_chunked_collection("podcast_chunks")
df = load_episodes()
memory = ConversationMemory()

print(f"✓ Collection loaded: {collection.count()} chunks")
print(f"✓ Episodes loaded: {len(df)}")
print(f"✓ Memory initialized")

# ═══════════════════════════════════════════════════════════
# Test 1: Single Query
# ═══════════════════════════════════════════════════════════

print("\n" + "="*60)
print("TEST 1: SINGLE QUERY")
print("="*60)

query = "I keep beating myself up over mistakes I made at work"

output = run_pipeline(
    user_query=query,
    collection=collection,
    memory=memory,
    df=df,
    top_k=10,              # Retrieve 10 chunks
    top_episodes=3,        # Show top 3 episodes
    search_method="hybrid",
    semantic_weight=0.7,
)

# Print formatted results
print_results(output)

# ═══════════════════════════════════════════════════════════
# Inspect Results Structure
# ═══════════════════════════════════════════════════════════

print("\n" + "="*60)
print("RESULTS STRUCTURE")
print("="*60)

print(f"\nTotal episodes found: {len(output['episodes'])}")
print(f"Total chunks retrieved: {output['total_chunks_retrieved']}")
print(f"Search method: {output['search_method']}")

print("\nFirst Episode Details:")
first_ep = output['episodes'][0]
print(f"  Episode ID: {first_ep['episode_id']}")
print(f"  Title: {first_ep['episode_title']}")
print(f"  Channel: {first_ep['show_name']}")
print(f"  Best Score: {first_ep['best_score']:.3f}")
print(f"  Chunks Retrieved: {first_ep['total_chunks_retrieved']}")

print("\n  Chunks:")
for i, chunk in enumerate(first_ep['chunks'], 1):
    meta = chunk['metadata']
    print(f"    {i}. {meta['start_time_display']} - {meta['end_time_display']} "
          f"(score: {chunk['similarity']:.3f})")

# ═══════════════════════════════════════════════════════════
# Test 2: Multiple Queries (Conversation)
# ═══════════════════════════════════════════════════════════

print("\n" + "="*60)
print("TEST 2: CONVERSATION FLOW")
print("="*60)

queries = [
    "I feel anxious about a job interview tomorrow",
    "What if I mess up and say something stupid?",
    "How do I stop overthinking?",
]

for i, query in enumerate(queries, 1):
    print(f"\n--- Turn {i} ---")
    print(f"Query: {query}")
    
    output = run_pipeline(
        user_query=query,
        collection=collection,
        memory=memory,
        df=df,
        top_k=10,
        top_episodes=2,  # Just top 2 for brevity
        search_method="hybrid",
    )
    
    print(f"Emotion: {output['emotional_context']['primary_emotion']}")
    print(f"Episodes: {[ep['episode_title'][:40] for ep in output['episodes']]}")

print(f"\n✓ Memory contains {len(memory)} turns")

# ═══════════════════════════════════════════════════════════
# Test 3: Compare Search Methods
# ═══════════════════════════════════════════════════════════

print("\n" + "="*60)
print("TEST 3: SEMANTIC vs HYBRID COMPARISON")
print("="*60)

memory.clear()  # Start fresh

query = "I need help setting boundaries with my family"

# Semantic only
print("\n--- Semantic Search ---")
output_semantic = run_pipeline(
    user_query=query,
    collection=collection,
    memory=memory,
    df=df,
    top_episodes=3,
    search_method="semantic",
)

semantic_episodes = [ep['episode_id'] for ep in output_semantic['episodes']]
print(f"Episodes: {semantic_episodes}")

memory.clear()

# Hybrid
print("\n--- Hybrid Search (70/30) ---")
output_hybrid = run_pipeline(
    user_query=query,
    collection=collection,
    memory=memory,
    df=df,
    top_episodes=3,
    search_method="hybrid",
    semantic_weight=0.7,
)

hybrid_episodes = [ep['episode_id'] for ep in output_hybrid['episodes']]
print(f"Episodes: {hybrid_episodes}")

print("\nComparison:")
print(f"  Same top result: {semantic_episodes[0] == hybrid_episodes[0]}")
print(f"  Overlap: {len(set(semantic_episodes) & set(hybrid_episodes))}/3")

# ═══════════════════════════════════════════════════════════
# Test 4: Verify Timestamps
# ═══════════════════════════════════════════════════════════

print("\n" + "="*60)
print("TEST 4: VERIFY TIMESTAMPS")
print("="*60)

memory.clear()

output = run_pipeline(
    user_query="I feel lost and don't know what direction to take",
    collection=collection,
    memory=memory,
    df=df,
    top_episodes=1,
)

ep = output['episodes'][0]
print(f"\nEpisode: {ep['episode_title']}")
print(f"\nTimestamp Verification:")

for chunk in ep['chunks']:
    meta = chunk['metadata']
    
    # Check all required timestamp fields exist
    required_fields = [
        'start_time_seconds',
        'end_time_seconds', 
        'duration_seconds',
        'start_time_display',
        'end_time_display',
        'duration_display'
    ]
    
    missing = [f for f in required_fields if f not in meta]
    
    if missing:
        print(f"  ✗ Missing fields: {missing}")
    else:
        print(f"  ✓ {meta['start_time_display']} - {meta['end_time_display']} ({meta['duration_display']})")
        
        # Verify YouTube URL with timestamp
        video_url = f"{ep['url']}&t={meta['start_time_seconds']}s"
        print(f"    URL: {video_url}")

print("\n" + "="*60)
print("✅ ALL TESTS COMPLETE")
print("="*60)

Loading resources...


⚠️ It looks like you upgraded from a version below 0.6 and could benefit from vacuuming your database. Run chromadb utils vacuum --help for more information.
INFO     | Loaded collection: podcast_chunks (1156 chunks)
INFO     | Pipeline started: 'I keep beating myself up over mistakes I made at w...'


✓ embedding type: <class 'list'> <class 'float'>
✓ Loaded 46 episodes
✓ Collection loaded: 1156 chunks
✓ Episodes loaded: 46
✓ Memory initialized

TEST 1: SINGLE QUERY


INFO     | Building BM25 index from chunks...
INFO     | ✓ BM25 index built for 1156 chunks
INFO     | Hybrid search returned 10 chunks (top score: 0.6531)
INFO     | Pipeline started: 'I keep beating myself up over mistakes I made at w...'
INFO     | Grouped into 5 episodes (avg 2.0 chunks per episode)
INFO     | Returning top 3 episodes
INFO     | Pipeline started: 'I feel anxious about a job interview tomorrow...'



🎯 EMOTIONAL PODCAST RECOMMENDATIONS

💭 Your Query: I keep beating myself up over mistakes I made at work
😢 Detected Emotion: shame
🔍 Search Method: hybrid
⚖️  Weights: 70% semantic, 30% keyword

📊 Found 3 episodes with 10 relevant segments

────────────────────────────────────────────────────────────
1. 🎧 Self-Compassion: How to Make it Work for You | Dr. Chris Germer, Being Well
   by Forrest Hanson
   Match Score: 0.653

   📍 Relevant Segments (6):

   1. ⏱️  53:33 - 56:07 (2m 33s)
      Score: 0.653
      Preview: try to put your fist through the wall because it's just not going to end very well for you most of t...

   2. ⏱️  5:37 - 7:59 (2m 22s)
      Score: 0.623
      Preview: for me to be able to access but man I'm just doing a normal job it's a hard job but it's a normal jo...

   3. ⏱️  55:47 - 58:09 (2m 22s)
      Score: 0.593
      Preview: our experience as it is now yeah it it it's actually a lot of work and it's also the most intelligen...

   4. ⏱️  57:55 - 59:59 (2m 3

...

Episodes: ['001']

--- Hybrid Search (70/30) ---


INFO     | Building BM25 index from chunks...
INFO     | ✓ BM25 index built for 1156 chunks
INFO     | Hybrid search returned 10 chunks (top score: 0.6603)
INFO     | Pipeline started: 'I need help setting boundaries with my family...'
INFO     | Grouped into 1 episodes (avg 10.0 chunks per episode)
INFO     | Returning top 1 episodes
INFO     | Pipeline started: 'I feel lost and don't know what direction to take...'


Episodes: ['001']

Comparison:
  Same top result: True
  Overlap: 1/3

TEST 4: VERIFY TIMESTAMPS


INFO     | Building BM25 index from chunks...
INFO     | ✓ BM25 index built for 1156 chunks
INFO     | Hybrid search returned 10 chunks (top score: 0.5253)
INFO     | Pipeline started: 'I feel lost and don't know what direction to take...'
INFO     | Grouped into 8 episodes (avg 1.2 chunks per episode)
INFO     | Returning top 1 episodes



Episode: It’s Not Too Late: How to Transform Your Life at Any Moment

Timestamp Verification:
  ✓ 35:21 - 37:55 (2m 34s)
    URL: https://www.youtube.com/watch?v=E9UU5N0k0XY&t=2121s

✅ ALL TESTS COMPLETE
